<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/1%EC%B0%A8_%ED%95%B4%EC%BB%A4%ED%86%A4_260312.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌽 옥수수 선물 거래 시뮬레이터 최종 설계

## 1단계 - 시나리오
- 서비스명: 🌾 FarmTrade 시뮬레이터
- 타겟: MZ세대 투자 입문자
- 핵심 시나리오: 가상 시드머니로 농산물 선물 거래 체험
- 기획 배경: 경제방송 PD 동생 + 전업 투자자 친구 리서치

## 2단계 - 알고리즘
- 초기 시드머니: 1,000만원
- 레버리지: 2x / 5x / 10x
- 가격 변동: 3초마다 ±3% 랜덤
- 수익 계산: 마진 × 레버리지 × 가격변동률
- 청산 조건: 손실이 마진 100% 도달 시 강제 청산

## 3단계 - 톤 & 문구
- 톤: 토스 느낌 (친근하고 캐주얼)
- 수익: 빨강 / 손실: 파랑 (한국 주식 스타일)
- 버튼: "매수할게요" / "청산할게요" / "다시 시작하기"

## 4단계 - UI 레이아웃
- 헤더: 잔고 표시 (수익률은 거래 통계 패널에 표시)
- 좌측 패널: 종목 선택, 롱/숏, 레버리지, 매수 버튼
- 우측 패널: 거래 내역 테이블
- 우측 상단: 4개 종목 실시간 가격 현황 카드

## 5단계 - 데이터 구조

### database.json
{
  "balance": 10000000,
  "trades": []
}

### data.js 종목 데이터
const ASSETS = [
  { id: "corn",    name: "옥수수", emoji: "🌽", price: 480  },
  { id: "wheat",   name: "밀",    emoji: "🌾", price: 620  },
  { id: "cattle",  name: "생우",  emoji: "🐄", price: 185  },
  { id: "soybean", name: "대두",  emoji: "🫘", price: 1420 }
]

- 🌽 옥수수 → 선물거래 기원, 스토리텔링 핵심
- 🌾 밀 → 우크라이나 전쟁, 뉴스에서 많이 나온 종목
- 🐄 생우 → 축산물 대표
- 🫘 대두 → 미국 농업 대표, 옥수수랑 단짝 종목

## 6~7단계 - 핵심 JS 로직
- setInterval → 3초마다 가격 랜덤 변동
- fetch POST → 거래 시 database.json에 저장
- fetch GET → 페이지 로드 시 거래 내역 불러오기
- DOM 조작 → 잔고/거래내역 테이블 실시간 업데이트

## 8단계 - 배포
- Express server.js → GET/POST API + 정적 파일 서빙
- Cloudflare Tunnel → 외부 공개 URL 생성
- GitHub → 전체 소스 업로드

## 파일 구조
```
/content/
├── server.js
├── database.json
├── templates/
│   └── index.html
└── static/
    └── data.js
```



---



In [18]:
!pkill -f "node /content/server.js"
!nohup node /content/server.js > /content/server.log 2>&1 &
!tail -n 5 /content/server.log



---



In [1]:
!mkdir -p /content/templates /content/static
!npm init -y
!npm install express

Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1",
    "start": "node server.js"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "dependencies": {
    "express": "^5.2.1"
  },
  "devDependencies": {},
  "description": ""
}



⠙⠙⠹⠸⠼⠴⠦⠧
up to date, audited 66 packages in 1s
⠧
⠧22 packages are looking for funding
⠧  run `npm fund` for details
⠧
found 0 vulnerabilities
⠧

In [2]:
%%writefile /content/database.json

{
  "balance": 10000000,
  "trades": []
}

Overwriting /content/database.json


In [3]:
%%writefile /content/static/data.js

const ASSETS = [
  { id: "corn",    name: "옥수수", emoji: "🌽", price: 480  },
  { id: "wheat",   name: "밀",    emoji: "🌾", price: 620  },
  { id: "cattle",  name: "생우",  emoji: "🐄", price: 185  },
  { id: "soybean", name: "대두",  emoji: "🫘", price: 1420 }
];

Overwriting /content/static/data.js


In [4]:
%%writefile /content/server.js

const express = require("express");
const fs = require("fs");
const path = require("path");
const app = express();

app.use(express.json());
app.use(express.static(path.join(__dirname, "static")));

// 메인 페이지
app.get("/", (req, res) => {
  res.sendFile(path.join(__dirname, "templates", "index.html"));
});

// 거래 내역 + 잔고 불러오기
app.get("/api/data", (req, res) => {
  const data = JSON.parse(fs.readFileSync("/content/database.json", "utf8"));
  res.json(data);
});

// 거래 저장
app.post("/api/trade", (req, res) => {
  const data = JSON.parse(fs.readFileSync("/content/database.json", "utf8"));
  const trade = req.body;
  data.trades.push(trade);
  data.balance = trade.balanceAfter;
  fs.writeFileSync("/content/database.json", JSON.stringify(data, null, 2));
  res.json({ success: true });
});

// 잔고 초기화
app.post("/api/reset", (req, res) => {
  const init = { balance: 10000000, trades: [] };
  fs.writeFileSync("/content/database.json", JSON.stringify(init, null, 2));
  res.json({ success: true });
});

app.listen(3000, () => console.log("서버 실행 중 - port 3000"));

Overwriting /content/server.js


In [20]:
%%writefile /content/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8">
  <!-- Pretendard: 토스 계열에서 쓰는 한국어 웹폰트 -->
  <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/orioncactus/pretendard/dist/web/static/pretendard.css">
  <!-- Chart.js: 수익/손실 막대차트 + 잔고 꺾은선 차트 -->
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
  <title>🌾 FarmTrade 시뮬레이터</title>
  <style>
    /* ── 전역 색상 변수 (토스 공식 컬러 기준) ── */
    :root {
      --bg: #f5f6f8;
      --card: #ffffff;
      --border: #e8eaed;
      --text: #202632;      /* Toss Gray */
      --muted: #8b95a1;
      --profit: #f04452;    /* 수익 = 빨간색 (한국 증시 관례) */
      --loss: #0064FF;      /* 손실 = 파란색 (한국 증시 관례) / Toss Blue */
      --primary: #202632;
      --accent: #0064FF;
    }

    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: 'Pretendard', 'Apple SD Gothic Neo', 'Noto Sans KR', sans-serif; background: var(--bg); color: var(--text); font-size: 14px; }
    /* 폼 요소는 별도 지정 안 하면 시스템 폰트로 렌더링되므로 명시 */
    input, button, select {
      font-family: 'Pretendard', 'Apple SD Gothic Neo', 'Noto Sans KR', sans-serif;
    }

    /* ── 헤더: sticky로 스크롤해도 상단 고정 ── */
    header {
      background: var(--card);
      border-bottom: 1px solid var(--border);
      padding: 0 40px;
      position: sticky;
      top: 0;
      z-index: 100;
      height: 60px;
      display: flex;
      align-items: center;
      justify-content: space-between;
    }
    .header-left {
      display: flex;
      align-items: center;
      gap: 10px;
    }
    .header-left img {
      height: 90px;
      width: auto;
    }
    .header-title {
      font-size: 16px;
      font-weight: 700;
      color: var(--text);
    }
    .header-right {
      display: flex;
      flex-direction: column;
      align-items: flex-end;
    }
    .header-right .label {
      font-size: 11px;
      color: var(--muted);
    }
    .header-right .amount {
      font-size: 22px;
      font-weight: 800;
      color: var(--text);
      letter-spacing: -0.5px;
    }

    /* ── 메인 그리드: 좌측 거래패널(320px 고정) + 우측 콘텐츠 ── */
    .main-grid {
     display: grid;
     grid-template-columns: 320px 1fr;
     gap: 16px;
     padding: 24px 40px;
     max-width: 1200px;
     margin: 0 auto;
     align-items: start;
    }

    /* ── 카드 공통 스타일 ── */
    .card {
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 24px;
    }
    .card-title {
      font-size: 13px;
      font-weight: 700;
      color: var(--muted);
      margin-bottom: 16px;
      letter-spacing: 0.3px;
    }

    /* ── 거래 패널: sticky로 스크롤해도 화면에 고정 ── */
    .trade-panel { grid-row: 1 / 4; position: sticky; top: 60px; align-self: start; }

    /* 종목 드롭다운: appearance:none으로 기본 화살표 제거 후 SVG로 커스텀 */
    select {
      width: 100%;
      padding: 11px 14px;
      border: 1px solid var(--border);
      border-radius: 8px;
      font-size: 14px;
      font-weight: 600;
      margin-bottom: 12px;
      cursor: pointer;
      background: var(--card);
      color: var(--text);
      appearance: none;
      background-image: url("data:image/svg+xml,%3Csvg width='10' height='6' viewBox='0 0 10 6' fill='none' xmlns='http://www.w3.org/2000/svg'%3E%3Cpath d='M1 1L5 5L9 1' stroke='%238b95a1' stroke-width='1.5' stroke-linecap='round'/%3E%3C/svg%3E");
      background-repeat: no-repeat;
      background-position: right 14px center;
    }

    /* 롱/숏 버튼: 선택 전 테두리만, 선택 후 배경색 채움 */
    .direction-btns { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; margin-bottom: 12px; }
    .btn-long  {
      background: #fff3f4; color: var(--profit);
      border: 1.5px solid var(--profit);
      border-radius: 8px; padding: 11px;
      font-weight: 700; font-size: 13px; cursor: pointer;
    }
    .btn-short {
      background: #f0f6ff; color: var(--loss);
      border: 1.5px solid var(--loss);
      border-radius: 8px; padding: 11px;
      font-weight: 700; font-size: 13px; cursor: pointer;
    }
    .btn-long.active  { background: var(--profit); color: white; }
    .btn-short.active { background: var(--loss);   color: white; }

    /* 레버리지 버튼: 선택 시 배경 검정으로 강조 */
    .leverage-btns { display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 8px; margin-bottom: 12px; }
    .leverage-btns button {
      padding: 10px; background: #f5f6f8;
      border: 1.5px solid var(--border);
      border-radius: 8px; font-weight: 700;
      font-size: 13px; cursor: pointer; color: var(--muted);
    }
    .leverage-btns button.active {
      background: var(--text); color: white;
      border-color: var(--text);
    }

    .margin-input { margin-bottom: 12px; }
    .margin-input input {
      width: 100%;
      padding: 11px 14px;
      border: 1px solid var(--border);
      border-radius: 8px;
      font-size: 14px;
      color: var(--text);
      background: var(--card);
    }
    .margin-input input:focus {
      outline: none;
      border-color: var(--accent);
    }

    /* 매수 버튼: disabled 상태(청산 중)일 때 회색, hover는 활성 상태에서만 */
    .btn-buy {
      width: 100%; padding: 14px;
      background: var(--accent);
      color: white; border: none;
      border-radius: 8px; font-size: 15px;
      font-weight: 700; cursor: pointer;
      margin-bottom: 12px;
    }
    .btn-buy:disabled { background: #8b95a1; cursor: not-allowed; }
    .btn-buy:hover:not(:disabled) { background: #1c6ef3; }
    .btn-reset {
      width: 100%; padding: 10px;
      background: #f5f6f8; color: var(--muted);
      border: none; border-radius: 8px;
      font-size: 13px; cursor: pointer;
    }

    /* ── 거래 내역 테이블 ── */
    .history-panel { overflow-x: auto; }
    table { width: 100%; border-collapse: collapse; font-size: 13px; }
    th {
      padding: 10px 12px; color: var(--muted);
      font-weight: 600; font-size: 12px;
      border-bottom: 1px solid var(--border);
      text-align: left; background: #fafbfc;
    }
    td { padding: 12px; border-bottom: 1px solid var(--border); color: var(--text); }
    tr:last-child td { border-bottom: none; }
    tr:hover td { background: #fafbfc; }
    .profit { color: var(--profit); font-weight: 700; }
    .loss   { color: var(--loss);   font-weight: 700; }

    /* ── 가격 현황 카드 (4종목 가로 배치) ── */
    .price-grid {
      grid-column: 2 / 3;
      grid-row: 1 / 2;
      display: grid;
      grid-template-columns: repeat(4, 1fr);
      gap: 12px;
    }
    /* 클릭 시 거래 패널 드롭다운이 해당 종목으로 자동 변경됨 */
    .price-card {
      background: var(--card);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 14px 12px;
      display: flex;
      align-items: center;
      gap: 10px;
      cursor: pointer;
      transition: box-shadow 0.15s;
    }
    .price-card:hover { box-shadow: 0 2px 12px rgba(0,0,0,0.08); }
    .price-card .emoji {
      font-size: 28px;
      background: #f5f6f8;
      border-radius: 50%;
      width: 48px; height: 48px;
      display: flex; align-items: center; justify-content: center;
      flex-shrink: 0;
    }
    .price-card .info { flex: 1; }
    .price-card .name  { font-size: 12px; color: var(--muted); margin-bottom: 4px; }
    .price-card .price { font-size: 18px; font-weight: 800; letter-spacing: -0.5px; }
    .price-card .change { font-size: 12px; margin-top: 2px; font-weight: 600; }
    /* ::before 로 "시작가 대비" 텍스트 자동 삽입 */
    .price-card .change-start::before { content: "시작가 대비 "; color: var(--muted); font-weight: 400; }
    .price-card .change-start { font-size: 11px; margin-top: 1px; white-space: nowrap; font-weight: 600; }

    /* ── 토스트 알림: 화면 하단 중앙에 2.5초 표시 ── */
    .toast {
      position: fixed;
      bottom: 32px;
      left: 50%;
      transform: translateX(-50%);
      background: #202632;
      color: white;
      padding: 12px 24px;
      border-radius: 8px;
      font-size: 14px;
      font-weight: 600;
      z-index: 9999;
      opacity: 0;
      transition: opacity 0.2s;
      pointer-events: none; /* 뒤 요소 클릭 막지 않게 */
    }
    .toast.show { opacity: 1; }

    /* ── 푸터 ── */
    footer {
      margin-top: 40px;
      border-top: 1px solid var(--border);
      padding: 24px 40px;
      background: var(--card);
    }
    .footer-links {
      display: flex; gap: 16px;
      font-size: 12px; color: var(--muted);
      margin-bottom: 12px; flex-wrap: wrap;
    }
    .footer-links a { color: var(--muted); text-decoration: none; }
    .footer-links a:hover { color: var(--text); }
    .footer-info { font-size: 11px; color: var(--muted); line-height: 1.8; }
    .footer-warning {
      margin-top: 8px; font-size: 11px;
      color: var(--muted); line-height: 1.6;
      padding: 10px 14px;
      background: #f5f6f8;
      border-radius: 6px;
    }
  </style>
</head>
<body>

<header>
  <div class="header-left">
    <img src="/toss.png" alt="토스">
    <span class="header-title">🌾 FarmTrade 시뮬레이터</span>
  </div>
  <div class="header-right">
    <div class="label">내 잔고</div>
    <div class="amount" id="balance">10,000,000원</div>
  </div>
</header>

<div class="main-grid">

  <!-- 거래 패널 -->
  <div class="card trade-panel">
    <div class="card-title">거래하기</div>

    <select id="asset-select">
      <option value="corn">🌽 옥수수</option>
      <option value="wheat">🌾 밀</option>
      <option value="cattle">🐄 생우</option>
      <option value="soybean">🫘 대두</option>
    </select>

    <div class="direction-btns">
      <button class="btn-long active"  id="btn-long"  onclick="setDirection('long')">▲ 롱 (상승 베팅)</button>
      <button class="btn-short"        id="btn-short" onclick="setDirection('short')">▼ 숏 (하락 베팅)</button>
    </div>

    <div class="leverage-btns">
      <button class="active" onclick="setLeverage(2,  this)">2x</button>
      <button               onclick="setLeverage(5,  this)">5x</button>
      <button               onclick="setLeverage(10, this)">10x</button>
    </div>

    <div class="margin-input">
      <input type="number" id="margin-input" placeholder="마진 입력 (원)" min="10000">
    </div>

    <!-- 잔고 비율로 마진 빠르게 채우기 -->
    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 6px; margin-bottom: 12px;">
      <button onclick="setMargin(0.1)"  style="padding: 7px; background: #f5f6f8; border: 1px solid var(--border); border-radius: 6px; font-size: 12px; font-weight: 600; color: var(--muted); cursor: pointer;">10%</button>
      <button onclick="setMargin(0.25)" style="padding: 7px; background: #f5f6f8; border: 1px solid var(--border); border-radius: 6px; font-size: 12px; font-weight: 600; color: var(--muted); cursor: pointer;">25%</button>
      <button onclick="setMargin(0.5)"  style="padding: 7px; background: #f5f6f8; border: 1px solid var(--border); border-radius: 6px; font-size: 12px; font-weight: 600; color: var(--muted); cursor: pointer;">50%</button>
      <button onclick="setMargin(1)"    style="padding: 7px; background: #f5f6f8; border: 1px solid var(--border); border-radius: 6px; font-size: 12px; font-weight: 600; color: var(--muted); cursor: pointer;">최대</button>
    </div>

    <button class="btn-buy"   onclick="executeTrade()">매수할게요</button>
    <button class="btn-reset" onclick="resetBalance()">다시 시작하기</button>

    <!-- 내 거래 통계: 거래할 때마다 실시간 업데이트 -->
    <div style="margin-top: 16px; padding-top: 16px; border-top: 1px solid var(--border);">
      <div class="card-title">내 거래 통계</div>
      <div style="display: flex; flex-direction: column; gap: 10px;">
        <div style="display: flex; justify-content: space-between;">
          <span style="color: var(--muted); font-size: 13px;">총 거래횟수</span>
          <span id="stat-count" style="font-weight: 700;">0건</span>
        </div>
        <div style="display: flex; justify-content: space-between;">
          <span style="color: var(--muted); font-size: 13px;">승률</span>
          <span id="stat-winrate" style="font-weight: 700;">-</span>
        </div>
        <div style="display: flex; justify-content: space-between;">
          <span style="color: var(--muted); font-size: 13px;">최대 수익</span>
          <span id="stat-maxwin" style="font-weight: 700; color: var(--profit);">-</span>
        </div>
        <div style="display: flex; justify-content: space-between;">
          <span style="color: var(--muted); font-size: 13px;">최대 손실</span>
          <span id="stat-maxloss" style="font-weight: 700; color: var(--loss);">-</span>
        </div>
        <div style="display: flex; justify-content: space-between;">
          <span style="color: var(--muted); font-size: 13px;">수익률</span>
          <span id="stat-return" style="font-weight: 700;">-</span>
        </div>
      </div>
    </div>
  </div>

  <!-- 거래 내역: 최근 10건만 표시 -->
  <div class="card history-panel">
    <div class="card-title">거래 내역</div>
    <table>
      <thead>
        <tr>
          <th>종목</th>
          <th>방향</th>
          <th>레버리지</th>
          <th>마진</th>
          <th>수수료</th>
          <th>수익/손실</th>
          <th>잔고</th>
          <th>시간</th>
        </tr>
      </thead>
      <tbody id="trade-tbody"></tbody>
    </table>
  </div>

  <!-- 차트: 수익/손실 막대 + 잔고 꺾은선 (최근 20건) -->
  <div class="card chart-panel" style="grid-column: 2 / 3; display: grid; grid-template-columns: 1fr 1fr; gap: 16px;">
   <div>
     <div class="card-title">수익/손실 히스토리</div>
     <canvas id="pnl-chart" height="160"></canvas>
   </div>
    <div>
     <div class="card-title">잔고 변화 추이</div>
     <canvas id="balance-chart" height="160"></canvas>
    </div>
  </div>

  <!-- 가격 현황: 3초마다 랜덤 변동, 클릭 시 해당 종목 자동 선택 -->
  <div class="price-grid">
    <div class="price-card" onclick="document.getElementById('asset-select').value='corn'">
      <div class="emoji">🌽</div>
      <div class="info">
        <div class="name">옥수수</div>
        <div class="price" id="p-corn">480</div>
        <div class="change" id="c-corn">-</div>
        <div class="change-start" id="s-corn">-</div>
      </div>
    </div>
    <div class="price-card" onclick="document.getElementById('asset-select').value='wheat'">
      <div class="emoji">🌾</div>
      <div class="info">
        <div class="name">밀</div>
        <div class="price" id="p-wheat">620</div>
        <div class="change" id="c-wheat">-</div>
        <div class="change-start" id="s-wheat">-</div>
      </div>
    </div>
    <div class="price-card" onclick="document.getElementById('asset-select').value='cattle'">
      <div class="emoji">🐄</div>
      <div class="info">
        <div class="name">생우</div>
        <div class="price" id="p-cattle">185</div>
        <div class="change" id="c-cattle">-</div>
        <div class="change-start" id="s-cattle">-</div>
      </div>
    </div>
    <div class="price-card" onclick="document.getElementById('asset-select').value='soybean'">
      <div class="emoji">🫘</div>
      <div class="info">
        <div class="name">대두</div>
        <div class="price" id="p-soybean">1420</div>
        <div class="change" id="c-soybean">-</div>
        <div class="change-start" id="s-soybean">-</div>
      </div>
    </div>
  </div>

</div>

<footer>
  <div class="footer-links">
    <a href="#">개인정보 처리방침</a>
    <a href="#">고객센터</a>
    <a href="#">공지사항</a>
    <a href="#">투자 유의사항</a>
    <a href="#">이용약관</a>
  </div>
  <div class="footer-info">
    사업자 등록번호: 000-00-00000 &nbsp;|&nbsp; 대표: 임소라 &nbsp;|&nbsp; 주소: 서울특별시 중구 청파로 463 (한국경제신문)
  </div>
  <div class="footer-warning">
    본 서비스는 한경 × 토스뱅크 부트캠프 해커톤 실습용 시뮬레이터입니다. 실제 투자 권유가 아니며, 제공되는 정보는 투자 판단의 참고용으로만 활용하시기 바랍니다.
  </div>
</footer>

<script src="/data.js"></script>
<script>
  // ── 차트 인스턴스 (한 번 생성 후 update()로 재사용) ──
  let pnlChart, balanceChart;

  // ── 차트용 누적 데이터 배열 ──
  let pnlData = [];
  let balanceData = [];

  // ── 거래 통계 변수 ──
  let winCount = 0;   // 수익 거래 횟수
  let tradeCount = 0; // 전체 거래 횟수
  let maxWin = 0;     // 최대 수익
  let maxLoss = 0;    // 최대 손실 (음수)

  // ── 현재 상태 ──
  let balance = 10000000;
  let direction = "long";
  let leverage = 2;

  // ── 현재 가격 (3초마다 랜덤 변동) ──
  let prices = { corn: 480, wheat: 620, cattle: 185, soybean: 1420 };

  // ── 시작 가격 (고정 — "시작가 대비" 계산에 사용) ──
  const initialPrices = { corn: 480, wheat: 620, cattle: 185, soybean: 1420 };

  // ── 페이지 로드 시 서버에서 저장된 잔고 + 거래내역 불러오기 ──
  fetch("/api/data")
    .then(r => r.json())
    .then(data => {
      balance = data.balance;
      updateBalance();
      renderTrades(data.trades);
    });

  // 헤더 잔고 숫자 갱신
  function updateBalance() {
    document.getElementById("balance").textContent =
      balance.toLocaleString() + "원";
  }

  // 롱/숏 버튼 활성화 전환
  function setDirection(dir) {
    direction = dir;
    document.getElementById("btn-long").classList.toggle("active",  dir === "long");
    document.getElementById("btn-short").classList.toggle("active", dir === "short");
  }

  // 레버리지 버튼 활성화 전환
  function setLeverage(val, el) {
    leverage = val;
    document.querySelectorAll(".leverage-btns button")
      .forEach(b => b.classList.remove("active"));
    el.classList.add("active");
  }

  // 잔고 비율로 마진 입력란 자동 채우기
  function setMargin(ratio) {
    document.getElementById("margin-input").value = Math.floor(balance * ratio);
  }

  // ── 3초마다 가격 랜덤 변동 (-3% ~ +3%) ──
  setInterval(() => {
    ASSETS.forEach(asset => {
      const change = (Math.random() * 6 - 3) / 100;
      const prev = prices[asset.id];
      prices[asset.id] = Math.round(prev * (1 + change) * 100) / 100;

      // 직전 대비 변동률
      const changeRate = ((prices[asset.id] - prev) / prev * 100).toFixed(2);
      document.getElementById("p-" + asset.id).textContent = prices[asset.id];
      const changeEl = document.getElementById("c-" + asset.id);
      changeEl.textContent = (changeRate > 0 ? "▲ " : "▼ ") + Math.abs(changeRate) + "%";
      changeEl.style.color = changeRate > 0 ? "var(--profit)" : "var(--loss)";

      // 시작가 대비 변동률
      const fromStart = ((prices[asset.id] - initialPrices[asset.id]) / initialPrices[asset.id] * 100).toFixed(2);
      const fromStartEl = document.getElementById("s-" + asset.id);
      if (fromStartEl) {
        fromStartEl.textContent = (fromStart > 0 ? "▲ " : "▼ ") + Math.abs(fromStart) + "%";
        fromStartEl.style.color = fromStart > 0 ? "var(--profit)" : "var(--loss)";
      }
    });
  }, 3000);

  // ── 매수 실행 ──
  function executeTrade() {
    const margin = parseInt(document.getElementById("margin-input").value);
    if (!margin || margin <= 0) { alert("마진을 입력해주세요."); return; }
    if (margin > balance)       { alert("잔고가 부족해요."); return; }

    const assetId    = document.getElementById("asset-select").value;
    const asset      = ASSETS.find(a => a.id === assetId);
    const entryPrice = prices[assetId]; // 진입 시점 가격 저장

    // 수수료: 마진의 0.1% (플랫폼 수익 모델)
    const fee = Math.round(margin * 0.001);
    balance -= margin;
    balance -= fee;
    updateBalance();

    // 토스트 알림 + 버튼 비활성화 (중복 클릭 방지)
    showToast(`${asset.emoji} ${asset.name} ${direction === "long" ? "롱" : "숏"} ${leverage}x 진입 — 3초 후 청산`);
    document.querySelector(".btn-buy").disabled = true;
    document.querySelector(".btn-buy").textContent = "청산 중...";

    // 3초 후 자동 청산
    setTimeout(() => {
      const exitPrice  = prices[assetId]; // 청산 시점 가격
      const changeRate = (exitPrice - entryPrice) / entryPrice;

      // 수익 계산: 마진 × 레버리지 × 가격변동률
      // 롱: 가격 오르면 수익 / 숏: 가격 내리면 수익
      const pnl = direction === "long"
        ? Math.round(margin * leverage * changeRate)
        : Math.round(margin * leverage * -changeRate);

      // 최대 손실 = 마진 전액 (잔고 음수 방지)
      const actualPnl = Math.max(pnl, -margin);
      balance += margin + actualPnl;
      updateBalance();

      const trade = {
        asset:        asset.emoji + " " + asset.name,
        direction:    direction === "long" ? "▲ 롱" : "▼ 숏",
        leverage:     leverage + "x",
        margin:       margin,
        fee:          fee,
        pnl:          actualPnl,
        balanceAfter: balance,
        time:         new Date().toLocaleTimeString()
      };

      // 서버 database.json에 거래 기록 저장
      fetch("/api/trade", {
        method: "POST",
        headers: { "Content-Type": "application/json" },
        body: JSON.stringify(trade)
      });

      addTradeRow(trade);

      // 버튼 복구
      document.querySelector(".btn-buy").disabled = false;
      document.querySelector(".btn-buy").textContent = "매수할게요";
    }, 3000);
  }

  // ── 거래 내역 테이블에 행 추가 ──
  function addTradeRow(trade) {
    const tbody = document.getElementById("trade-tbody");
    const tr = document.createElement("tr");
    tr.innerHTML = `
      <td>${trade.asset}</td>
      <td style="color: ${trade.direction.includes('롱') ? '#f04452' : '#0064FF'}; font-weight: 700;">${trade.direction}</td>
      <td>${trade.leverage}</td>
      <td>${trade.margin.toLocaleString()}원</td>
      <td style="color: var(--muted);">-${trade.fee.toLocaleString()}원</td>
      <td class="${trade.pnl >= 0 ? "profit" : "loss"}">
        ${trade.pnl >= 0 ? "+" : ""}${trade.pnl.toLocaleString()}원
      </td>
      <td>${trade.balanceAfter.toLocaleString()}원</td>
      <td>${trade.time}</td>
    `;
    tbody.prepend(tr); // 최신 거래가 위에 오도록 prepend

    // 통계 + 차트 업데이트
    tradeCount++;
    if (trade.pnl > 0) winCount++;
    if (trade.pnl > maxWin) maxWin = trade.pnl;
    if (trade.pnl < maxLoss) maxLoss = trade.pnl;
    updateStats();

    pnlData.push(trade.pnl);
    balanceData.push(trade.balanceAfter);
    updateCharts();

    // 테이블은 최근 10건만 유지
    const tbodyLimit = document.getElementById("trade-tbody");
    if (tbodyLimit.rows.length > 10) {
      tbodyLimit.deleteRow(tbodyLimit.rows.length - 1);
    }
  }

  // 서버에서 불러온 기존 거래내역 렌더링 (최신순으로 뒤집어서 추가)
  function renderTrades(trades) {
    trades.forEach(addTradeRow);
  }

  // ── 다시 시작하기: 잔고/내역/차트/통계 전부 초기화 ──
  // [버그 수정] 통계 초기화를 .then() 안으로 이동 (밖에 있으면 리셋 전에 실행됨)
  function resetBalance() {
    if (!confirm("정말 다시 시작할까요? 거래 내역이 모두 사라져요.")) return;
    fetch("/api/reset", { method: "POST" })
      .then(() => {
        balance = 10000000;
        updateBalance();
        document.getElementById("trade-tbody").innerHTML = "";
        pnlData = [];
        balanceData = [];
        winCount = 0; tradeCount = 0; maxWin = 0; maxLoss = 0;
        updateCharts();
        updateStats();
      });
  }

  // ── 차트 업데이트 (최근 20건만 표시) ──
  function updateCharts() {
    const recentPnl     = pnlData.slice(-20);
    const recentBalance = balanceData.slice(-20);
    // x축 라벨 숨김 (흐름만 표시)
    const labels = recentPnl.map(() => "");

    if (pnlChart) {
      // 이미 차트 있으면 데이터만 교체 후 update()
      pnlChart.data.labels = labels;
      pnlChart.data.datasets[0].data = recentPnl;
      pnlChart.data.datasets[0].backgroundColor = recentPnl.map(v => v >= 0 ? "#f04452" : "#0064FF");
      pnlChart.data.datasets[0].borderColor     = recentPnl.map(v => v >= 0 ? "#f04452" : "#0064FF");
      pnlChart.update();
    } else {
      // 첫 거래 시 차트 인스턴스 생성
      pnlChart = new Chart(document.getElementById("pnl-chart"), {
        type: "bar",
        data: {
          labels,
          datasets: [{
            label: "수익/손실 (원)",
            data: recentPnl,
            backgroundColor: recentPnl.map(v => v >= 0 ? "#f04452" : "#0064FF"),
            borderColor:     recentPnl.map(v => v >= 0 ? "#f04452" : "#0064FF"),
          }]
        },
        options: {
          responsive: true,
          plugins: { legend: { display: false } },
          scales: {
            x: { ticks: { font: { size: 11 } } },
            y: { ticks: { callback: v => (v / 10000).toLocaleString() + "만 원" } }
          }
        }
      });
    }

    if (balanceChart) {
      balanceChart.data.labels = labels;
      balanceChart.data.datasets[0].data = recentBalance;
      balanceChart.update();
    } else {
      balanceChart = new Chart(document.getElementById("balance-chart"), {
        type: "line",
        data: {
          labels,
          datasets: [{
            label: "잔고 (원)",
            data: recentBalance,
            borderColor: "#0064FF",
            backgroundColor: "rgba(0,100,255,0.08)",
            tension: 0.3,
            fill: true,
            pointRadius: 0, // 점 없애서 꺾은선만 표시
          }]
        },
        options: {
          responsive: true,
          plugins: { legend: { display: false } },
          scales: {
            x: { ticks: { font: { size: 11 } } },
            y: { ticks: { callback: v => (v / 10000).toLocaleString() + "만 원" } }
          }
        }
      });
    }
  }

  // ── 거래 통계 패널 업데이트 ──
  function updateStats() {
    // 수익률: (현재잔고 - 시작잔고) / 시작잔고 × 100
    const returnRate = ((balance - 10000000) / 10000000 * 100).toFixed(2);
    document.getElementById("stat-count").textContent    = tradeCount + "건";
    document.getElementById("stat-winrate").textContent  = tradeCount ? Math.round(winCount / tradeCount * 100) + "%" : "-";
    document.getElementById("stat-maxwin").textContent   = maxWin  ? "+" + maxWin.toLocaleString()  + "원" : "-";
    document.getElementById("stat-maxloss").textContent  = maxLoss ? maxLoss.toLocaleString() + "원" : "-";
    const retEl = document.getElementById("stat-return");
    retEl.textContent = (returnRate > 0 ? "+" : "") + returnRate + "%";
    retEl.style.color = returnRate >= 0 ? "var(--profit)" : "var(--loss)";
  }

  // ── 토스트 알림: 2.5초 후 자동으로 사라짐 ──
  function showToast(msg) {
    const t = document.getElementById("toast");
    t.textContent = msg;
    t.classList.add("show");
    setTimeout(() => t.classList.remove("show"), 2500);
  }

</script>

<div class="toast" id="toast"></div>

</body>
</html>

Overwriting /content/templates/index.html


In [6]:
!nohup node /content/server.js > /content/server.log 2>&1 &
!tail -n 5 /content/server.log

In [7]:
!npm install -g cloudflared
!nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 1 package in 2s
⠼

In [14]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" /content/tunnel.log | tail -n 1

https://fonts-hardwood-accommodations-las.trycloudflare.com




---



In [15]:
!cat /content/database.json

{
  "balance": 9820910,
  "trades": [
    {
      "asset": "🌽 옥수수",
      "direction": "▲ 롱",
      "leverage": "10x",
      "margin": 5000000,
      "fee": 5000,
      "pnl": 898905,
      "balanceAfter": 10893905,
      "time": "오후 2:19:35"
    },
    {
      "asset": "🐄 생우",
      "direction": "▼ 숏",
      "leverage": "10x",
      "margin": 5446952,
      "fee": 5447,
      "pnl": -798777,
      "balanceAfter": 10089681,
      "time": "오후 2:19:41"
    },
    {
      "asset": "🫘 대두",
      "direction": "▼ 숏",
      "leverage": "10x",
      "margin": 10089681,
      "fee": 10090,
      "pnl": 242822,
      "balanceAfter": 10322413,
      "time": "오후 2:19:49"
    },
    {
      "asset": "🌽 옥수수",
      "direction": "▲ 롱",
      "leverage": "10x",
      "margin": 5161206,
      "fee": 5161,
      "pnl": -180472,
      "balanceAfter": 10136780,
      "time": "오후 2:19:59"
    },
    {
      "asset": "🌽 옥수수",
      "direction": "▲ 롱",
      "leverage": "10x",
      "margin": 5161206,
      